## Data Profiling and Stats with Azure Databricks

### Create the Unity Catalog Structure

In [ ]:
%sql

CREATE SCHEMA IF NOT EXISTS food_delivery_lab.bronze
COMMENT 'Raw ingested delivery data';

CREATE SCHEMA IF NOT EXISTS food_delivery_lab.silver
COMMENT 'Cleaned and transformed data';

CREATE VOLUME IF NOT EXISTS food_delivery_lab.bronze.raw_files
COMMENT 'Landing zone for incoming CSV files';

### Write CSV Files to UC Volume

In [ ]:
import requests

# Define the current catalog
catalog_name = "food_delivery_lab"

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/bronze/raw_files"

# List of files to download
files = ["orders.csv", "restaurants.csv", "payments.csv", "executives.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DP-750/refs/heads/main/Data_Profiling_and_Operations/Data/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

print("CSV Files written successfully")

### Load into Bronze Delta Tables

In [ ]:
def load_csv(path, table):
    (spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(path)
        .write.mode("overwrite")
        .saveAsTable(table))

base = "/Volumes/food_delivery_lab/bronze/raw_files"

load_csv(f"{base}/orders.csv",      "food_delivery_lab.bronze.orders")
load_csv(f"{base}/restaurants.csv", "food_delivery_lab.bronze.restaurants")
load_csv(f"{base}/executives.csv",     "food_delivery_lab.bronze.executives")
load_csv(f"{base}/payments.csv",    "food_delivery_lab.bronze.payments")

print("All tables loaded")

### Collect Statistics with `ANALYZE TABLE`

In [ ]:
%sql

-- Collect basic table statistics (row count and size)
ANALYZE TABLE food_delivery_lab.bronze.orders COMPUTE STATISTICS;

In [ ]:
%sql

-- Collect statistics for specific columns
ANALYZE TABLE food_delivery_lab.bronze.orders COMPUTE STATISTICS FOR COLUMNS 
    order_value_usd, items_count;


In [ ]:
%sql

-- Collect statistics for all columns
ANALYZE TABLE food_delivery_lab.bronze.orders COMPUTE STATISTICS FOR ALL COLUMNS;

In [ ]:
%sql

-- View column-level statistics
DESC EXTENDED food_delivery_lab.bronze.orders order_value_usd;

### View Statistics with `DESCRIBE TABLE`

In [ ]:
%sql

-- The following displays collected statistics alongside the table metadata
DESCRIBE TABLE EXTENDED food_delivery_lab.bronze.orders;

### Generate Summary using the `DataFrame API`

In [ ]:
# Generate summary statistics for numeric columns using the DataFrame API
df = spark.table("food_delivery_lab.bronze.orders")
df.describe().show()